In [1]:
!pip install /kaggle/input/iterative-stratification/iterative_stratification-0.1.9-py3-none-any.whl
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA

Processing /kaggle/input/iterative-stratification/iterative_stratification-0.1.9-py3-none-any.whl


In [2]:
train_features = pd.read_csv("/kaggle/input/lish-moa/train_features.csv")
train_targets  = pd.read_csv("/kaggle/input/lish-moa/train_targets_scored.csv")
test_features  = pd.read_csv("/kaggle/input/lish-moa/test_features.csv")

In [3]:
train_targets

,sig_id,5-alpha_reductase_inhibitor,11-beta-hsd1_inhibitor,acat_inhibitor,acetylcholine_receptor_agonist,acetylcholine_receptor_antagonist,acetylcholinesterase_inhibitor,adenosine_receptor_agonist,adenosine_receptor_antagonist,adenylyl_cyclase_activator,...,tropomyosin_receptor_kinase_inhibitor,trpv_agonist,trpv_antagonist,tubulin_inhibitor,tyrosine_kinase_inhibitor,ubiquitin_specific_protease_inhibitor,vegfr_inhibitor,vitamin_b,vitamin_d_receptor_agonist,wnt_inhibitor
0,id_000644bb2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,id_000779bfc,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,id_000a6266a,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,id_0015fd391,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,id_001626bd3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23809,id_fffb1ceed,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23810,id_fffb70c0c,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23811,id_fffc1c3f4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23812,id_fffcb9e7c,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Preprocessing

Train has samples cp_type = ctl_vehicle (control experiments). They always have all targets = 0, so learning from them is pointless.

In [4]:
mask = train_features['cp_type'] != 'ctl_vehicle'

train_features_trt = train_features[mask].reset_index(drop=True)
train_targets_trt  = train_targets[mask].reset_index(drop=True)

In [5]:
train_features_trt.head(5)

,sig_id,cp_type,cp_time,cp_dose,g-0,g-1,g-2,g-3,g-4,g-5,...,c-90,c-91,c-92,c-93,c-94,c-95,c-96,c-97,c-98,c-99
0,id_000644bb2,trt_cp,24,D1,1.0620,0.5577,-0.2479,-0.6208,-0.1944,-1.0120,...,0.2862,0.2584,0.8076,0.5523,-0.1912,0.6584,-0.3981,0.2139,0.3801,0.4176
1,id_000779bfc,trt_cp,72,D1,0.0743,0.4087,0.2991,0.0604,1.0190,0.5207,...,-0.4265,0.7543,0.4708,0.0230,0.2957,0.4899,0.1522,0.1241,0.6077,0.7371
2,id_000a6266a,trt_cp,48,D1,0.6280,0.5817,1.5540,-0.0764,-0.0323,1.2390,...,-0.7250,-0.6297,0.6103,0.0223,-1.3240,-0.3174,-0.6417,-0.2187,-1.4080,0.6931
3,id_0015fd391,trt_cp,48,D1,-0.5138,-0.2491,-0.2656,0.5288,4.0620,-0.8095,...,-2.0990,-0.6441,-5.6300,-1.3780,-0.8632,-1.2880,-1.6210,-0.8784,-0.3876,-0.8154
4,id_001626bd3,trt_cp,72,D2,-0.3254,-0.4009,0.9700,0.6919,1.4180,-0.8244,...,0.0042,0.0048,0.6670,1.0690,0.5523,-0.3031,0.1094,0.2885,-0.3786,0.7125


### Categorical columns

In [6]:
cat_cols = ['cp_time', 'cp_dose']
train_cat = pd.get_dummies(train_features_trt[cat_cols], columns=cat_cols, prefix=cat_cols)
test_cat  = pd.get_dummies(test_features[cat_cols], columns=cat_cols, prefix=cat_cols)

In [7]:
train_cat

,cp_time_24,cp_time_48,cp_time_72,cp_dose_D1,cp_dose_D2
0,True,False,False,True,False
1,False,False,True,True,False
2,False,True,False,True,False
3,False,True,False,True,False
4,False,False,True,False,True
...,...,...,...,...,...
21943,False,False,True,True,False
21944,True,False,False,False,True
21945,True,False,False,False,True
21946,True,False,False,True,False


### Numerical features (g- and c-)

In [8]:
scaler = StandardScaler()

num_cols = [c for c in train_features.columns if c.startswith("g-") or c.startswith("c-")]

In [9]:
train_num_scaled = pd.DataFrame(scaler.fit_transform(train_features_trt[num_cols]), columns=num_cols)
test_num_scaled  = pd.DataFrame(scaler.transform(test_features[num_cols]), columns=num_cols)

In [10]:
train_num_scaled

,g-0,g-1,g-2,g-3,g-4,g-5,g-6,g-7,g-8,g-9,...,c-90,c-91,c-92,c-93,c-94,c-95,c-96,c-97,c-98,c-99
0,0.549598,0.795008,-0.401914,-0.735073,-0.273020,-0.734098,-1.187307,0.151066,0.441443,-0.178297,...,0.391971,0.365039,0.660978,0.514686,0.171632,0.631496,0.055513,0.363935,0.484324,0.531958
1,-0.142510,0.609207,0.126195,-0.021846,0.912782,0.561238,0.254855,0.476366,-0.091567,0.677683,...,0.046710,0.600043,0.497959,0.271068,0.390410,0.531001,0.314322,0.312714,0.604833,0.753503
2,0.245482,0.824935,1.337754,-0.165078,-0.114606,1.168297,0.182982,0.369311,0.136858,1.022054,...,-0.097895,-0.055827,0.565480,0.270746,-0.337366,0.049518,-0.059054,0.117185,-0.462432,0.722993
3,-0.554608,-0.211058,-0.419002,0.468577,3.886571,-0.562958,-2.263102,0.337379,0.059864,-1.057065,...,-0.763516,-0.062651,-2.454968,-0.373763,-0.130316,-0.529359,-0.519624,-0.259100,0.077845,-0.323019
4,-0.422591,-0.400350,0.773924,0.639345,1.302707,-0.575551,-0.335397,0.047969,-0.354989,0.688850,...,0.255359,0.244859,0.592924,0.752505,0.505707,0.058046,0.294193,0.406486,0.082611,0.736445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21943,-0.081898,-1.209766,0.083714,-0.319513,-0.320612,0.480866,-0.147793,0.023427,-0.282273,0.029524,...,0.291546,0.410248,0.297089,0.415913,0.043978,0.089477,-0.105990,0.721796,0.057566,0.411165
21944,-0.096893,0.020258,-0.269935,-0.616970,-0.543622,0.729757,0.648892,0.448480,-0.129286,0.020536,...,0.348710,0.255000,-0.122994,0.418537,0.498922,0.045224,0.272418,0.765032,0.561469,0.566906
21945,-1.123739,0.533266,-0.523948,0.951984,-0.784515,0.680740,-0.272480,-0.309790,0.417223,0.516074,...,0.460955,0.452330,0.290555,0.113427,-0.105782,-0.345545,0.144730,0.172113,0.139318,0.498188
21946,0.272110,0.389364,0.261456,0.128925,0.750656,0.092189,0.023163,0.220472,0.372060,-0.819646,...,0.199793,0.444369,0.172696,0.329798,0.941869,0.662331,0.324198,0.642056,-0.049970,0.293701


### Final data

In [11]:
train_num_scaled["sig_id"] = train_features_trt["sig_id"].values
train_cat["sig_id"]        = train_features_trt["sig_id"].values
test_num_scaled["sig_id"]  = test_features["sig_id"].values
test_cat["sig_id"]         = test_features["sig_id"].values

In [12]:
Xtr = train_num_scaled.merge(train_cat, on="sig_id")
Xte = test_num_scaled.merge(test_cat, on="sig_id")
Xte.head()

,g-0,g-1,g-2,g-3,g-4,g-5,g-6,g-7,g-8,g-9,...,c-96,c-97,c-98,c-99,sig_id,cp_time_24,cp_time_48,cp_time_72,cp_dose_D1,cp_dose_D2
0,-0.577032,0.262422,-0.658341,0.376439,1.431705,-0.017763,-0.259621,0.375116,-0.047849,1.665764,...,0.252618,0.213295,0.363021,-0.295005,id_0004d9e33,True,False,False,True,False
1,-0.322738,0.388866,1.003704,-0.558546,-0.439935,-0.159323,-2.174697,0.533544,-0.062736,0.020718,...,0.259625,-0.029804,-0.448136,-0.263802,id_001897cda,False,False,True,True,False
2,-0.064800,-0.075511,-0.540168,0.052074,-1.488335,0.328657,-0.403137,-0.102277,0.518992,0.435270,...,0.650120,0.821443,0.543784,0.107728,id_002429b5b,True,False,False,True,False
3,0.143737,0.343351,0.206714,0.359268,-0.655224,-0.894673,0.674725,0.021492,0.653480,0.172430,...,0.625147,0.167493,0.581959,-0.160414,id_00276f245,True,False,False,False,True
4,-0.473394,-1.481608,1.684355,0.130286,-0.656104,0.107148,0.574838,0.739649,0.278290,-1.104277,...,0.103437,-0.061118,0.808893,1.516880,id_0027f1083,False,True,False,True,False


In [13]:
target_cols = [c for c in train_targets_trt.columns if c != "sig_id"]
df_train = Xtr.merge(train_targets, on="sig_id", how="inner")
df_train.shape


(21948, 1084)

In [14]:
feature_cols = [c for c in Xtr.columns if c != "sig_id"]
len(feature_cols)

877

In [15]:
feature_cols = [c for c in Xtr.columns if c != "sig_id"] 

X_train = df_train[feature_cols].astype("float32").values
y_train = df_train[target_cols].astype("float32").values
X_test  = Xte[feature_cols].astype("float32").values

X_train.shape, y_train.shape, X_test.shape

((21948, 877), (21948, 206), (3982, 877))

# Model

In [16]:
class MoAModel(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 2048),
            nn.BatchNorm1d(2048),
            nn.GELU(),
            nn.Dropout(0.5),

            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(256, out_dim) 
        )
    def forward(self, x): 
        return self.net(x)

# Validation splits

Why MultilabelStratifiedKFold
- This is a multi-label task: each sample can have several "1" s in 206 columns.

- This algorithm takes into account the entire label matrix at once.

- It tries to make sure that in each fold, the proportion of "1" for each label is as close as possible to the global one.

- This means that even rare MoAs will be present in both train and valid.

In [17]:
SEED = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)

def mean_columnwise_logloss(y_true, y_prob):
    y_prob = np.clip(y_prob, 1e-15, 1 - 1e-15)
    return log_loss(y_true, y_prob)

In [18]:
class MoADataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32)
    def __len__(self): 
        return self.X.shape[0]
    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]             
        return self.X[idx], self.y[idx] 

In [19]:
@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total, n = 0.0, 0
    preds, trues = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE) 
        yb = yb.to(DEVICE).float()
        logits = model(xb)
        loss = criterion(logits, yb)
        total += loss.item() * xb.size(0)
        n += xb.size(0)
        preds.append(torch.sigmoid(logits).cpu().numpy())
        trues.append(yb.cpu().numpy())
    y_pred = np.concatenate(preds, axis=0)
    y_true = np.concatenate(trues, axis=0)
    return total/n, mean_columnwise_logloss(y_true, y_pred)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb = xb.to(DEVICE) 
        yb = yb.to(DEVICE).float()
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item() * xb.size(0)
        n += xb.size(0)
    return total/n


In [20]:
class EarlyStopping:
    def __init__(self, patience=5):
        self.best = float('inf')
        self.wait = 0
        self.state = None
        self.patience = patience

    def step(self, value, model):
        if value < self.best:
            self.best = value
            self.wait = 0
            self.state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            return True 
        else:
            self.wait += 1
            return self.wait < self.patience

In [21]:
def add_pca_features(X_tr, X_va, g_idx, c_idx, n_comp_g=350, n_comp_c=30, seed=42):
    """
    Додає PCA-компоненти для g- та c- фіч у train і val.
    Фітимо PCA лише на train, трансформимо і train, і val.
    """
    Xg_tr, Xc_tr = X_tr[:, g_idx], X_tr[:, c_idx]
    Xg_va, Xc_va = X_va[:, g_idx], X_va[:, c_idx]

    # --- скейлінг ---
    scaler_g = StandardScaler()
    scaler_c = StandardScaler()
    Xg_tr_s = scaler_g.fit_transform(Xg_tr)
    Xc_tr_s = scaler_c.fit_transform(Xc_tr)
    Xg_va_s = scaler_g.transform(Xg_va)
    Xc_va_s = scaler_c.transform(Xc_va)

    # --- PCA ---
    pca_g = PCA(n_components=n_comp_g, random_state=seed)
    pca_c = PCA(n_components=n_comp_c, random_state=seed)

    Xg_tr_p = pca_g.fit_transform(Xg_tr_s)
    Xc_tr_p = pca_c.fit_transform(Xc_tr_s)
    Xg_va_p = pca_g.transform(Xg_va_s)
    Xc_va_p = pca_c.transform(Xc_va_s)

    X_tr_new = np.concatenate([X_tr, Xg_tr_p, Xc_tr_p], axis=1)
    X_va_new = np.concatenate([X_va, Xg_va_p, Xc_va_p], axis=1)

    return X_tr_new, X_va_new, scaler_g, scaler_c, pca_g, pca_c


In [22]:
BATCH_TRAIN = 256
BATCH_VALID = 512
EPOCHS = 30
FOLDS = 5
LR = 1e-3
WD = 1e-4

mskf = MultilabelStratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_scores = []
models = []

feature_cols = [c for c in Xtr.columns if c != "sig_id"]
g_idx = [i for i,c in enumerate(feature_cols) if c.startswith("g-")]
c_idx = [i for i,c in enumerate(feature_cols) if c.startswith("c-")]

for fold, (tr_idx, va_idx) in enumerate(mskf.split(X_train, y_train), 1):
    print(f"\n===== FOLD {fold}/{FOLDS} =====")
    X_tr_raw, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_va_raw, y_va = X_train[va_idx], y_train[va_idx]
    
    X_tr, X_va, scaler_g, scaler_c, pca_g, pca_c = add_pca_features(
        X_tr_raw, X_va_raw, g_idx, c_idx,
        n_comp_g=350, n_comp_c=30, seed=SEED
    )

    train_loader = DataLoader(MoADataset(X_tr, y_tr), batch_size=BATCH_TRAIN, shuffle=True,  num_workers=0, pin_memory=True)
    valid_loader = DataLoader(MoADataset(X_va, y_va), batch_size=BATCH_VALID, shuffle=False, num_workers=0, pin_memory=True)

    model = MoAModel(in_dim=X_tr.shape[1], out_dim=y_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    criterion = nn.BCEWithLogitsLoss()

    es = EarlyStopping(patience=5)

    for epoch in range(1, EPOCHS + 1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_mcll = validate(model, valid_loader, criterion)
        print(f"Epoch {epoch:02d} | train {tr_loss:.4f} | valid {va_loss:.4f} | mcll {va_mcll:.5f}")

        if not es.step(va_mcll, model):
            print("Early stopping")
            break

    model.load_state_dict(es.state)
    models.append({
        "model": model,
        "scaler_g": scaler_g,
        "scaler_c": scaler_c,
        "pca_g": pca_g,
        "pca_c": pca_c
    })
    fold_scores.append(es.best)
    print(f"[FOLD {fold}] best mcll: {es.best:.5f}")

print("\nCV mean mcll:", float(np.mean(fold_scores)))



===== FOLD 1/5 =====
Epoch 01 | train 0.1872 | valid 0.0317 | mcll 3.83845
Epoch 02 | train 0.0279 | valid 0.0232 | mcll 3.57013
Epoch 03 | train 0.0227 | valid 0.0214 | mcll 3.41534
Epoch 04 | train 0.0213 | valid 0.0204 | mcll 3.30151
Epoch 05 | train 0.0204 | valid 0.0197 | mcll 3.18631
Epoch 06 | train 0.0196 | valid 0.0190 | mcll 3.06424
Epoch 07 | train 0.0190 | valid 0.0185 | mcll 2.99153
Epoch 08 | train 0.0184 | valid 0.0181 | mcll 2.92682
Epoch 09 | train 0.0180 | valid 0.0178 | mcll 2.87877
Epoch 10 | train 0.0176 | valid 0.0176 | mcll 2.83923
Epoch 11 | train 0.0172 | valid 0.0174 | mcll 2.80376
Epoch 12 | train 0.0169 | valid 0.0174 | mcll 2.78652
Epoch 13 | train 0.0166 | valid 0.0173 | mcll 2.77208
Epoch 14 | train 0.0163 | valid 0.0173 | mcll 2.75453
Epoch 15 | train 0.0160 | valid 0.0171 | mcll 2.73611
Epoch 16 | train 0.0157 | valid 0.0171 | mcll 2.73038
Epoch 17 | train 0.0154 | valid 0.0171 | mcll 2.73711
Epoch 18 | train 0.0150 | valid 0.0171 | mcll 2.72700
Epoch 

# Submit

In [23]:
def transform_test_with_fold(X_test_raw, g_idx, c_idx, scaler_g, scaler_c, pca_g, pca_c):
    Xg_te = X_test_raw[:, g_idx]
    Xc_te = X_test_raw[:, c_idx]
    Xg_te_s = scaler_g.transform(Xg_te)
    Xc_te_s = scaler_c.transform(Xc_te)
    Xg_te_p = pca_g.transform(Xg_te_s)
    Xc_te_p = pca_c.transform(Xc_te_s)
    return np.concatenate([X_test_raw, Xg_te_p, Xc_te_p], axis=1)

In [24]:
X_test_raw = X_test

fold_preds = []

with torch.no_grad():
    for pack in models:
        m = pack["model"].to(DEVICE).eval()

        X_te_ext = transform_test_with_fold(
            X_test_raw, g_idx, c_idx,
            pack["scaler_g"], pack["scaler_c"],
            pack["pca_g"], pack["pca_c"]
        )

        te_loader = DataLoader(MoADataset(X_te_ext), batch_size=1024,
                               shuffle=False, num_workers=0, pin_memory=True)

        probs_all = []
        for xb in te_loader:
            xb = xb.to(DEVICE)
            logits = m(xb)
            probs_all.append(torch.sigmoid(logits).cpu().numpy())

        fold_preds.append(np.vstack(probs_all))

preds = np.mean(fold_preds, axis=0)
preds.shape

(3982, 206)

In [25]:
ctl_mask = (test_features["cp_type"].values == "ctl_vehicle")
preds[ctl_mask, :] = 0.0

In [26]:
submission = pd.DataFrame(preds, columns=target_cols)
submission.insert(0, "sig_id", test_features["sig_id"].values)
submission.to_csv("submission.csv", index=False)

submission.shape, submission.head(3)


((3982, 207),
          sig_id  5-alpha_reductase_inhibitor  11-beta-hsd1_inhibitor  \
 0  id_0004d9e33                     0.000885                0.000632   
 1  id_001897cda                     0.001472                0.001304   
 2  id_002429b5b                     0.000000                0.000000   
 
    acat_inhibitor  acetylcholine_receptor_agonist  \
 0        0.002061                        0.019246   
 1        0.002909                        0.003892   
 2        0.000000                        0.000000   
 
    acetylcholine_receptor_antagonist  acetylcholinesterase_inhibitor  \
 0                           0.025004                        0.006007   
 1                           0.002780                        0.003904   
 2                           0.000000                        0.000000   
 
    adenosine_receptor_agonist  adenosine_receptor_antagonist  \
 0                    0.003808                       0.006968   
 1                    0.003638                    